# Assignment-4

**UCS420 “Cognitive Computing”**

**A Cognitive FAQ System Using Pandas**

**Adarsh Sharma**

**Roll No: 1024170359**

**Batch: 3Q33**

Cognitive Computing Assignment Submission

## Q.1 Build Your Personalized Knowledge Base

Take the roll number, extract the last two digits, and build a 6-row FAQ DataFrame (4 fixed entries + 2 personalized entries based on `category = ["billing", "account", "general"][d % 3]`).

In [ ]:
import pandas as pd

roll_number = "1024170359"
last_two_digits = [int(d) for d in roll_number[-2:]]
categories = ["billing", "account", "general"]

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
     "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
     "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
     "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.",
     "keywords": "pay payment upi fee", "category": "billing"},
]

personalized_entries = [
    {"question": "what are your holiday timings",
     "answer": "We remain closed on all national holidays and weekends.",
     "keywords": "holiday timing schedule", "category": categories[last_two_digits[0] % 3]},
    {"question": "how do i get a refund for an overpayment",
     "answer": "Refunds are processed within 5-7 business days to your original payment method.",
     "keywords": "refund overpayment money", "category": categories[last_two_digits[1] % 3]},
]

faq_entries = fixed_entries + personalized_entries
faq_df = pd.DataFrame(faq_entries)
faq_df

## Q.2 Generate and Score a Hypothesis

Implement a scoring function that takes a query string and returns all matching entries ranked by confidence (number of overlapping words between the query and each entry's question/keywords).

In [ ]:
def score_query(query, df):
    query_words = set(query.lower().split())
    results = []
    for idx, row in df.iterrows():
        keyword_words = set(row["keywords"].lower().split())
        question_words = set(row["question"].lower().split())
        overlap = query_words & (keyword_words | question_words)
        score = len(overlap)
        if score > 0:
            results.append({"question": row["question"], "answer": row["answer"],
                             "category": row["category"], "score": score})
    ranked = sorted(results, key=lambda x: x["score"], reverse=True)
    return ranked

results = score_query("how can i pay my fee", faq_df)
for r in results:
    print(r)

## Q.3 same_category Function

Write a function `same_category(category_name, df)` that returns all questions belonging to a given category. Call it using the category of one of the personalized entries from Q1 (`general`).

In [ ]:
def same_category(category_name, df):
    return df[df["category"] == category_name][["question", "answer", "keywords", "category"]]

print(same_category("general", faq_df))

## Q.4 Update an Entry and Save to CSV

Pick one entry in the knowledge base, ask the user for a new keyword, add it to that entry's keywords, and save the entire updated DataFrame to `1024170359_faq_data.csv`.

In [ ]:
new_keyword = input("Enter a new keyword to add to the first FAQ entry: ")
faq_df.loc[0, "keywords"] = faq_df.loc[0, "keywords"] + " " + new_keyword

faq_df.to_csv("1024170359_faq_data.csv", index=False)
print("Updated DataFrame saved as 1024170359_faq_data.csv")
faq_df

## Q.5 Group By Category

Using `groupby`, print how many FAQ entries exist per category.

In [ ]:
category_counts = faq_df.groupby("category").size()
print(category_counts)

## Q.6 Handle Ties in Scoring

Modify the Q.2 scoring function so that if two or more entries tie for the highest score, it prints all matching entries instead of silently picking one. Demonstrate with one query that produces a tie (matching both "fee" entries) and one that doesn't.

In [ ]:
def score_query_with_ties(query, df):
    query_words = set(query.lower().split())
    results = []
    for idx, row in df.iterrows():
        keyword_words = set(row["keywords"].lower().split())
        question_words = set(row["question"].lower().split())
        overlap = query_words & (keyword_words | question_words)
        score = len(overlap)
        if score > 0:
            results.append({"question": row["question"], "answer": row["answer"],
                             "category": row["category"], "score": score})

    if not results:
        print("No matching entries found.")
        return []

    max_score = max(r["score"] for r in results)
    top_matches = [r for r in results if r["score"] == max_score]

    if len(top_matches) > 1:
        print(f"Tie detected! {len(top_matches)} entries share the highest score of {max_score}:")
        for r in top_matches:
            print(r)
    else:
        print("Best match:")
        print(top_matches[0])

    return top_matches

print("Query: 'fee'")
score_query_with_ties("fee", faq_df)

print()

print("Query: 'password'")
score_query_with_ties("password", faq_df)